# ReAct Abstention Benchmark (RAB) v0.1

This notebook runs the synthetic baseline and analyses the answer/abstain risk-coverage trade-off. It works in Kaggle and Google Colab. The core benchmark is CPU-only; no GPU or model training is required.

**Important:** the CSV is self-generated synthetic data. It is not a real-agent or LLM evaluation dataset.

## 1. Obtain the project code

The cell below clones the GitHub repository unless the files are already present.

In [ ]:
from pathlib import Path
if not Path('run_benchmark.py').exists():
    !git clone https://github.com/MHKhan50/react-abstention-benchmark.git
    %cd react-abstention-benchmark
else:
    print('Project files already present:', Path.cwd())

In [ ]:
!pip -q install -r requirements-notebook.txt
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 2. Use the Kaggle dataset if attached; otherwise regenerate it

On Kaggle the attached dataset is used. In Colab the CSV from the repository is used, or the same fixed-seed data is regenerated.

In [ ]:
kaggle_candidates = list(Path('/kaggle/input').rglob('rab_synthetic_episodes.csv')) if Path('/kaggle/input').exists() else []
local_csv = Path('data/rab_synthetic_episodes.csv')
if kaggle_candidates:
    DATA_CSV = kaggle_candidates[0]
    print('Using attached Kaggle dataset:', DATA_CSV)
elif local_csv.exists():
    DATA_CSV = local_csv
    print('Using repository dataset:', DATA_CSV)
else:
    DATA_CSV = None
    print('No attached CSV found; the benchmark will generate the self-generated dataset.')

In [ ]:
if DATA_CSV is None:
    !python run_benchmark.py --episodes-per-seed 4000 --seeds 17 29 43 71 101
else:
    !python run_benchmark.py --input-data {DATA_CSV} --out-dir outputs

## 3. Inspect pooled policy metrics

In [ ]:
metrics = pd.read_csv('outputs/policy_metrics.csv')
display(metrics[['policy', 'unsupported_answer_rate', 'correct_abstention_rate', 'answerable_coverage', 'unnecessary_abstention_rate', 'overall_coverage', 'selective_risk']])

## 4. Threshold sensitivity and risk-coverage plot

In [ ]:
sweep = pd.read_csv('outputs/threshold_sweep.csv')
display(sweep[['threshold', 'unsupported_answer_rate', 'answerable_coverage', 'overall_coverage', 'selective_risk']])
plt.figure(figsize=(7, 5))
plt.plot(sweep['overall_coverage'], sweep['selective_risk'], marker='o')
plt.xlabel('Overall answer coverage')
plt.ylabel('Selective risk')
plt.title('RAB synthetic threshold risk-coverage curve')
plt.grid(True)
plt.show()

## 5. Condition-level behaviour

Check whether presence-only answers on irrelevant/contradictory evidence and how threshold policies change behaviour.

In [ ]:
condition_metrics = pd.read_csv('outputs/condition_metrics.csv')
display(condition_metrics[['condition', 'policy', 'overall_coverage', 'unsupported_answer_rate', 'correct_abstention_rate']])

# Formula vs simulation: the largest gap shows how far results depend on the chosen score ranges
!python src/rab/expected.py
display(pd.read_csv('outputs/expected_vs_observed.csv').sort_values('abs_diff', ascending=False).head(10))

## Interpretation guardrails

- Report the seed-level results and variation; do not cherry-pick a seed.
- Treat threshold-sweep findings as conditional on the synthetic score distributions.
- Do not describe these results as trained-model performance, real-world agent safety, or empirical hallucination rates.
- A stronger next study needs real tool traces, independent evidence annotations, semantic relevance/contradiction checks, and external validation.